# Feature Engineering & Preprocessing

Covers:
- Transaction frequency/velocity features
- Time-based features: `hour_of_day`, `day_of_week`, `time_since_signup`
- Scaling (StandardScaler) and one-hot encoding
- Class imbalance handling via SMOTE (training set only)
- Outputs processed train/test splits for both datasets

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

fraud = pd.read_csv('../data/processed/fraud_with_country.csv', parse_dates=['signup_time', 'purchase_time'])
cc = pd.read_csv('../data/processed/creditcard_cleaned.csv')

## 1. Fraud_Data — Feature Engineering

In [ ]:
# Time features
fraud['hour_of_day'] = fraud['purchase_time'].dt.hour
fraud['day_of_week'] = fraud['purchase_time'].dt.dayofweek
fraud['time_since_signup'] = (fraud['purchase_time'] - fraud['signup_time']).dt.total_seconds() / 3600  # hours

# Transaction velocity: count of transactions per user_id
freq = fraud.groupby('user_id')['user_id'].transform('count')
fraud['user_tx_count'] = freq

# Transaction frequency within 24h window per user
fraud_sorted = fraud.sort_values(['user_id', 'purchase_time']).copy()
fraud_sorted['tx_last_24h'] = fraud_sorted.groupby('user_id')['purchase_time'].transform(
    lambda x: x.expanding().count() - 1  # cumulative count minus self
)

fraud = fraud_sorted
fraud[['user_id', 'purchase_time', 'hour_of_day', 'day_of_week', 'time_since_signup', 'user_tx_count', 'tx_last_24h']].head()

## 2. Fraud_Data — Encoding & Scaling

In [ ]:
drop_cols = ['user_id', 'signup_time', 'purchase_time', 'device_id', 'ip_address', 'ip_int']
cat_cols  = ['source', 'browser', 'sex', 'country']
target    = 'class'

fraud_model = fraud.drop(columns=drop_cols, errors='ignore')
fraud_model = pd.get_dummies(fraud_model, columns=cat_cols, drop_first=True)

X = fraud_model.drop(columns=[target])
y = fraud_model[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print(f'Train shape: {X_train_scaled.shape} | Test shape: {X_test_scaled.shape}')

## 3. Class Imbalance — SMOTE

**Justification**: SMOTE (Synthetic Minority Over-sampling Technique) is preferred over random undersampling because:
- Undersampling discards majority-class information, which is costly with limited data.
- SMOTE synthesizes realistic minority samples via interpolation, preserving dataset size and improving minority-class boundary learning.
- Applied **only on the training set** to prevent data leakage from the test set.

In [ ]:
print('Class distribution BEFORE SMOTE:')
print(y_train.value_counts())
print(f'Fraud rate: {y_train.mean():.2%}')

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print('\nClass distribution AFTER SMOTE:')
print(pd.Series(y_train_res).value_counts())
print(f'Fraud rate: {y_train_res.mean():.2%}')

## 4. Save Fraud_Data Processed Splits

In [ ]:
X_train_res_df = pd.DataFrame(X_train_res, columns=X_train_scaled.columns)
X_train_res_df['class'] = y_train_res
X_train_res_df.to_csv('../data/processed/fraud_train.csv', index=False)

X_test_scaled['class'] = y_test.values
X_test_scaled.to_csv('../data/processed/fraud_test.csv', index=False)

print('Saved: fraud_train.csv, fraud_test.csv')

## 5. CreditCard — Scaling & SMOTE

In [ ]:
X_cc = cc.drop(columns=['Class'])
y_cc = cc['Class']

X_cc_train, X_cc_test, y_cc_train, y_cc_test = train_test_split(
    X_cc, y_cc, test_size=0.2, random_state=42, stratify=y_cc
)

scaler_cc = StandardScaler()
X_cc_train_scaled = pd.DataFrame(scaler_cc.fit_transform(X_cc_train), columns=X_cc_train.columns)
X_cc_test_scaled  = pd.DataFrame(scaler_cc.transform(X_cc_test), columns=X_cc_test.columns)

print('Class distribution BEFORE SMOTE:')
print(y_cc_train.value_counts())

smote_cc = SMOTE(random_state=42)
X_cc_train_res, y_cc_train_res = smote_cc.fit_resample(X_cc_train_scaled, y_cc_train)

print('\nClass distribution AFTER SMOTE:')
print(pd.Series(y_cc_train_res).value_counts())

In [ ]:
cc_train_df = pd.DataFrame(X_cc_train_res, columns=X_cc_train_scaled.columns)
cc_train_df['Class'] = y_cc_train_res
cc_train_df.to_csv('../data/processed/creditcard_train.csv', index=False)

X_cc_test_scaled['Class'] = y_cc_test.values
X_cc_test_scaled.to_csv('../data/processed/creditcard_test.csv', index=False)

print('Saved: creditcard_train.csv, creditcard_test.csv')